In [2]:
import numpy as np
import pandas as pd

In [3]:
np.random.seed(42)

In [4]:
# ── 1. Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv("flow_logs_data.csv")
df["synthetic"] = False          # mark all original rows

def _short(val):
    """Extract the VM short name from a resource path, lowercase."""
    if pd.isna(val) or not str(val).strip():
        return None
    return str(val).split("/")[-1].lower()

df["_src"] = df["SrcVm"].apply(_short)
df["_dst"] = df["DestVm"].apply(_short)

In [5]:
# ── 2. Task 1: AFD → web1  (mirror of AFD → web2) ────────────────────────────
# "AFD traffic" = Allowed HTTP/HTTPS flows with no SrcVm (Front Door service tag)
afd_web2 = df[
    (df["_dst"] == "flowdemo-web2") &
    (df["FlowStatus"] == "Allowed") &
    (df["L7Protocol"].isin(["www-http", "https"])) &
    df["SrcVm"].isna()
].copy()

afd_web1 = afd_web2.copy()
afd_web1["DestVm"]    = "rg-jxd-ddd/flowdemo-web1"
afd_web1["DestIp"]    = "10.0.1.5"          # new node — mirrors web2's subnet
afd_web1["synthetic"] = True
afd_web1["_dst"]      = "flowdemo-web1"

print(f"[Task 1] AFD→web1 synthetic rows created: {len(afd_web1)}")
print(f"         BytesSrcToDest total: {afd_web1['BytesSrcToDest'].sum():,.0f}"
      f"  (equals AFD→web2: {afd_web2['BytesSrcToDest'].sum():,.0f})")

[Task 1] AFD→web1 synthetic rows created: 16
         BytesSrcToDest total: 2,857,197  (equals AFD→web2: 2,857,197)


In [6]:
# ── 2b. Task 1b: web1 → applb  (mirror of web2 → applb) ──────────────────────
web2_applb = df[
    (df["_src"] == "flowdemo-web2") &
    (df["_dst"] == "flowdemo-applb")
].copy()

web1_applb = web2_applb.copy()
web1_applb["SrcVm"]     = "rg-jxd-ddd/flowdemo-web1"
web1_applb["SrcIp"]     = "10.0.1.5"
web1_applb["synthetic"] = True
web1_applb["_src"]      = "flowdemo-web1"

print(f"[Task 1b] web1→applb synthetic rows: {len(web1_applb)}")

[Task 1b] web1→applb synthetic rows: 3


In [7]:
# ── 3. Task 2: applb → app2 / app3 / app4 ────────────────────────────────────
app1_rows     = df[(df["_src"] == "flowdemo-applb") & (df["_dst"] == "flowdemo-app1")].copy()
app1_s2d_total = app1_rows["BytesSrcToDest"].sum()   # 50% baseline
app1_d2s_total = app1_rows["BytesDestToSrc"].sum()

# target share relative to app1's 50%
targets = {
    "app2": {"share": 30 / 50, "dest_vm": "rg-jxd-ddd/flowdemo-app2", "dest_ip": "10.0.2.11"},
    "app3": {"share": 10 / 50, "dest_vm": "rg-jxd-ddd/flowdemo-app3", "dest_ip": "10.0.2.12"},
    "app4": {"share": 10 / 50, "dest_vm": "rg-jxd-ddd/flowdemo-app4", "dest_ip": "10.0.2.13"},
}

applb_synth_parts = []
print(f"\n[Task 2] applb baseline (app1): {len(app1_rows)} rows | "
      f"BytesSrcToDest={app1_s2d_total:,.0f} = 50%")

for appname, cfg in targets.items():
    n_rows    = max(1, round(len(app1_rows) * cfg["share"]))
    templates = app1_rows.sample(n=n_rows, replace=True, random_state=42).copy()

    # Scale byte counts so totals exactly match the target share
    scale_s2d = (app1_s2d_total * cfg["share"]) / templates["BytesSrcToDest"].sum()
    scale_d2s = (app1_d2s_total * cfg["share"]) / templates["BytesDestToSrc"].sum()
    templates["BytesSrcToDest"] = (templates["BytesSrcToDest"] * scale_s2d).round(1)
    templates["BytesDestToSrc"] = (templates["BytesDestToSrc"] * scale_d2s).round(1)

    templates["DestVm"]    = cfg["dest_vm"]
    templates["DestIp"]    = cfg["dest_ip"]
    templates["synthetic"] = True
    templates["_dst"]      = f"flowdemo-{appname}"

    print(f"         {appname}: {n_rows} row(s) | "
          f"BytesSrcToDest={templates['BytesSrcToDest'].sum():,.1f} "
          f"= {cfg['share']*50:.0f}%")
    applb_synth_parts.append(templates)

applb_synth = pd.concat(applb_synth_parts, ignore_index=True)


[Task 2] applb baseline (app1): 3 rows | BytesSrcToDest=22,737 = 50%
         app2: 2 row(s) | BytesSrcToDest=13,642.2 = 30%
         app3: 1 row(s) | BytesSrcToDest=4,547.4 = 10%
         app4: 1 row(s) | BytesSrcToDest=4,547.4 = 10%


In [8]:
# ── 4. Combine and verify ─────────────────────────────────────────────────────
all_synth = pd.concat([afd_web1, web1_applb, applb_synth], ignore_index=True)
final     = pd.concat([df, all_synth], ignore_index=True).drop(columns=["_src", "_dst"])

print(f"\nOriginal rows  : {len(df):,}")
print(f"Synthetic rows : {len(all_synth):,}")
print(f"Total rows     : {len(final):,}")

# Quick ratio check
_f = final.copy()
_f["_src"] = _f["SrcVm"].apply(_short)
_f["_dst"] = _f["DestVm"].apply(_short)
applb_totals = _f[_f["_src"] == "flowdemo-applb"].groupby("_dst")["BytesSrcToDest"].sum()
grand        = applb_totals.sum()
print("\napplb traffic distribution (all rows):")
for node, val in applb_totals.sort_values(ascending=False).items():
    print(f"  {node:<30} {val:>10,.1f} bytes  ({val/grand:.0%})")

final.head()


Original rows  : 1,476
Synthetic rows : 23
Total rows     : 1,499

applb traffic distribution (all rows):
  flowdemo-app1                    22,737.0 bytes  (50%)
  flowdemo-app2                    13,642.2 bytes  (30%)
  flowdemo-app3                     4,547.4 bytes  (10%)
  flowdemo-app4                     4,547.4 bytes  (10%)


,TimeGenerated,SrcVm,SrcIp,DestVm,DestIp,DestPort,L7Protocol,FlowStatus,BytesSrcToDest,BytesDestToSrc,synthetic
0,2026-06-20 19:54:17.799372+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,2026-06-20 19:54:16.917128+00:00,NaN,NaN,rg-jxd-ddd/flowdemo-web2,10.0.1.4,3620.0,ep-pcp,Denied,0.0,0.0,False
2,2026-06-20 19:54:16.917128+00:00,NaN,NaN,rg-jxd-ddd/flowdemo-web2,10.0.1.4,6466.0,Unknown,Denied,0.0,0.0,False
3,2026-06-20 19:54:16.917128+00:00,NaN,NaN,rg-jxd-ddd/flowdemo-pgprimary,10.0.3.4,28975.0,Unknown,Allowed,102.0,54.0,False
4,2026-06-20 19:54:16.917128+00:00,NaN,NaN,rg-jxd-ddd/flowdemo-web2,10.0.1.4,6496.0,Unknown,Denied,0.0,0.0,False


In [9]:
# ── 5. Save ───────────────────────────────────────────────────────────────────
final.to_csv("flow_logs_data_synthetic.csv", index=False)
print("\nSaved: flow_logs_data_synthetic.csv")
print(f"synthetic=True rows: {final['synthetic'].sum()}")
print(f"synthetic=False rows: {(~final['synthetic']).sum()}")


Saved: flow_logs_data_synthetic.csv
synthetic=True rows: 23
synthetic=False rows: 1476
